# 02a — La ventana: por qué rango móvil de 120 s, y qué dicen los papers

Esta es la rama de la **representación temporal**. La columna vertebral está en
`02_supervisado.ipynb`; acá se justifica una sola decisión: qué transformación
se aplica antes de entrenar, y cómo se compara con los dos papers del dataset.

La respuesta corta:

- Entre transformaciones causales se elige por **especificidad en el escenario
  benigno no visto (S0)**, no por AUC. El rango móvil de 120 s gana ahí (0,523)
  y `zscore` —que tiene el AUC más alto, 0,900— pierde (0,419).
- El rango móvil quita el *nivel* de operación, no la *estructura* temporal: la
  correlación con el reloj queda en ~0,23, igual que en el crudo.
- Contra los papers: **no se reproduce el 0,760 publicado.** Nuestra réplica de
  su configuración da 0,898 **con** el reloj y 0,783 **sin** él. El desvío es el
  reloj, y el número publicado es compatible con una vista sin él.

Ese último punto hay que decirlo con cuidado: no "superamos al paper".
Coincidimos en magnitud cuando se le quita el reloj.


## Antes de empezar: si lo abrís en Google Colab

Este notebook vive en el repo y lee `datasets/`, `src/`, `resultados/` y `modelos/` desde la raíz del proyecto. En Colab esos archivos no existen, así que la celda de abajo clona el repo en `/content/data4cyber` y se para ahí.

**Cómo abrirlo:**

1. En Colab: *Archivo → Abrir notebook → GitHub*, pegá la URL del repo y elegí el notebook.
2. Ejecutá la primera celda. Clona el repo (una sola vez por sesión) y queda listo para correr.

Fuera de Colab la celda no hace nada.


In [ ]:
# --- Preparar el entorno: solo hace algo si estás en Google Colab ---
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/maxiricalde/data4cyber.git"
DESTINO = Path("/content/data4cyber")

try:
    import google.colab  # noqa: F401

    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    # CatBoost no viene preinstalado en Colab; lo instalamos una sola vez.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"], check=False)
    # Una sola vez por sesión: clonar el repo y pararse en su raíz.
    if not (DESTINO / "src" / "data.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(DESTINO)], check=True)
    os.chdir(DESTINO)
    print("proyecto en:", os.getcwd())
else:
    print("corriendo local: no hay nada que preparar")


corriendo local: no hay nada que preparar


In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# El notebook vive en notebooks/, pero los datasets y src/ están en la raíz del proyecto.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src.data import build_matrices  # solo se usa para quitar la capa de anotación
from src.features import detrend
from src.reduce import block_pca, corr_filter, dimensionless_columns, physical_features

pd.set_option("display.width", 140)
print("raíz del proyecto:", RAIZ)
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, StratifiedKFold


raíz del proyecto: /media/windows/Users/pablo/Documents/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab


## 0. Los datos

Cada archivo CSV corresponde a un escenario. Los archivos comparten las mismas columnas: mediciones eléctricas, estado de la batería y variables del perfil del escenario (`Profile.*`). Más adelante veremos qué información aportan estas últimas y por qué hay que interpretarlas con cuidado.


In [3]:
ESCENARIOS = ("S0", "S1", "S1alt", "S2", "S3", "S4", "S5", "S6")

frames = pd.concat(
    [pd.read_csv(f"datasets/dataset{nombre}.csv").assign(scenario=nombre) for nombre in ESCENARIOS],
    ignore_index=True,
)

print("filas totales:", frames.shape)
print(frames["scenario"].value_counts().sort_index())

filas totales: (14354, 157)
scenario
S0       1798
S1       1798
S1alt    1792
S2       1768
S3       1800
S4       1798
S5       1800
S6       1800
Name: count, dtype: int64


`build_matrices` separa las columnas que usaremos para aprender de las anotaciones que sirven para describir los datos. Entre las que se apartan están `attack_active`, `attack_phase`, `timestamp` y el precio de MQTT; el precio se excluye porque puede revelar directamente la etiqueta del ataque. Además excluimos `Profile.*` — las siete columnas de contexto reproducido — por la fuga que mostramos recién arriba.

La función devuelve:

- `X`: las 132 variables de entrada (también llamadas *features*), ya sin `Profile.*`.
- `y_binary`: la respuesta correcta de cada fila: `1` significa ataque y `0`, benigno.
- `scenario`: el escenario al que pertenece cada fila.

In [4]:
# Sin Profile.*: la fuga de identidad de escenario que mostramos arriba.
m = build_matrices(frames, include_profile=False)

X_crudo = m.X
y = m.y_binary
escenario = m.scenario

print("X:", X_crudo.shape)
print(f"ataques: {int(y.sum()):,} ({100 * y.mean():.1f} %)")

X: (14354, 132)
ataques: 7,586 (52.8 %)


## 1. ¿Por qué rango móvil de 120 s?

La decisión tiene tres partes y conviene separarlas: **si** transformar, **con
qué ventana** y **con qué transformación**. Las tres se resuelven con el mismo
criterio, y ese criterio hay que declararlo antes de mirar los números.


### Lo que hace el paper, y qué hacemos nosotros

El paper del dataset (Data4Cyber) entrena sobre los valores **crudos a paso de 1 s**: describe el procesamiento como *"default step size: 1 s"* y construye las tablas de features sobre esa grilla fija, sin ninguna ventana móvil ni transformación temporal. Nosotros, en cambio, convertimos cada variable en su **rango móvil de 120 s** (máximo menos mínimo dentro de la ventana).

¿Por qué no usar el crudo directamente, como hace el paper? La celda siguiente lee `resultados/barrido_ventana.csv` (generado por `python -m src.sweep`), que barre métodos y ventanas sobre las mismas 132 variables y el mismo Random Forest, en una versión rápida de 150 árboles. Lo que importa es el orden relativo, no el número exacto. La tabla que sigue compara los cinco métodos causales en la misma ventana de 120 s. La distinción entre los dos papers y la comparación completa están en §3.4.


### La regla de decisión

> Entre transformaciones causales se ordena por **especificidad en S0** —el único
> escenario benigno que el modelo nunca ve—, porque es la métrica operativa. Un
> detector que marca todo tiene AUC alto y es inútil. Desempate: **recall@5** en
> ataques y **robustez numérica**.

El criterio no es invento nuestro: `src/sweep.py` lo declara en su docstring
(*"spec_S0: ... This is the number that decides usability"*). El AUC mide
separación *dentro* de cada fold; no mide transferencia a un benigno no visto.


In [5]:
barrido = pd.read_csv("resultados/barrido_ventana.csv")

filas = barrido[barrido["method"].isin(["none", "diff"]) | (barrido["window"] == 120.0)]
orden = ["none", "mean", "zscore", "std", "range", "diff"]
tabla_metodos = (
    filas.assign(método=filas["method"])
    .set_index("método")
    .reindex([m for m in orden if m in set(filas["method"])])
    [["mean_abs_rho", "auc_mean", "spec_S0"]]
    .rename(columns={"mean_abs_rho": "|rho| reloj", "auc_mean": "AUC medio", "spec_S0": "espec. S0"})
)
display(tabla_metodos.round(3))


,|rho| reloj,AUC medio,espec. S0
método,,,
none,0.227,0.824,0.455
mean,0.077,0.886,0.466
zscore,0.059,0.900,0.419
std,0.245,0.893,0.502
range,0.230,0.880,0.523
diff,0.009,0.640,0.223


La lectura en tres números:

1. **Especificidad en S0** (el escenario benigno que el modelo no vio): **0,455 con crudo contra 0,523 con rango móvil**. La mejora es modesta, y conviene notar qué pasó: con `Profile.*` adentro, el rango móvil *parecía* arreglar S0 (0,466 → 0,948), pero eso era la fuga — `Profile.*` está congelado en S0, su rango da 0 y S0 queda "limpio". Sin `Profile.*`, el rango móvil ayuda poco (0,455 → 0,523): el nivel de las variables eléctricas también cambia entre escenarios, pero ya no hay un atajo de "congelado = benigno".

2. **AUC medio**: sube de 0,824 a 0,880. El rango móvil deja mejor ordenados los puntajes, aunque su beneficio en S0 sea chico.

3. **Correlación con el reloj** (|Spearman| medio contra la posición relativa): ~0,23 en ambos. El rango móvil elimina el *nivel*, no la *estructura* temporal.

Elegimos el rango móvil de 120 s porque es causal y corta (una fila depende solo de los 120 s anteriores) y porque elimina el nivel de operación, que es lo que no se traslada entre escenarios. Pero su efecto en la especificidad de S0, sin `Profile.*`, es modesto: la transferencia sigue siendo el límite central del dataset.

**¿Y cuánto importa la ventana?** Barrido con la receta de producción (`block_pca` + Random Forest, LOSO, sin `Profile.*`). La celda siguiente lo lee de `resultados/blockpca_ventana.csv`, generado con el mismo protocolo que el resto del cuaderno:


In [6]:
ventana_bp = pd.read_csv("resultados/blockpca_ventana.csv").set_index("entrada")
display(
    ventana_bp.rename(columns={"auc_mean": "AUC", "spec_S0": "espec. S0", "recall_5": "recall@5 %"})
    .drop(columns="window")
    .round(3)
)


,AUC,espec. S0,recall@5 %
entrada,,,
crudo 1 s,0.868,0.445,0.376
rango 20 s,0.876,0.406,0.441
rango 30 s,0.888,0.409,0.451
rango 60 s,0.896,0.454,0.451
rango 120 s,0.905,0.503,0.564


Dos cosas para leer de ahí. Primero, una ventana de 20 s es **peor que el crudo en especificidad S0** (0,406 contra 0,445): la ventana corta no alcanza a borrar el nivel y sí mete ruido. Segundo, el beneficio crece hasta 120 s: el AUC pasa de 0,868 a 0,905, la especificidad de 0,445 a 0,503 y el recall@5 de 0,376 a 0,564. Por eso no elegimos 20 s por parecernos al paper: si se quiere imitar al paper, conviene el crudo; si se usa ventana, 120 s es la mejor de las probadas.

El promedio usa los siete folds con ataque (incluye S3, que en nuestros datos sí tiene ambas clases). La comparación con el paper, en las secciones 3 y 5, usa solo los seis folds que el paper reporta.


**¿Y con los 141 canales del paper?** Acá hay que separar dos cosas que se llaman igual.

GRIDGUARD-AI usa **ventanas de 20 pasos**: su unidad de evaluación es un recorte de 20 segundos × 141 variables, y con eso obtiene 14.202 ventanas de las 14.354 filas (14.354 − 8 × 19 = 14.202, así que descarta las primeras 19 filas de cada escenario). La ventana es el **objeto que se clasifica**, no un filtro: los valores adentro quedan crudos, con su nivel. El TCN consume la secuencia completa; del RF el paper solo dice que entrega un score tabular, no detalla cómo aplana la ventana.

Nuestro **rango móvil de 20 s** es otra operación: seguimos con una fila por muestra y cada canal se reduce a su máximo menos mínimo de los últimos 20 s. Eso *borra el nivel* —y con el nivel, el reloj— en vez de conservarlo.

Con esa distinción, `python -m src.paper_window` mide el efecto del detrend sobre las columnas del paper; no replica su unidad de muestreo. La tabla:


In [7]:
# Barrido de ventana sobre los 141 canales del paper: lo genera
# `python -m src.paper_window`, que reusa el RF y el protocolo LOSO de
# src/compare_paper.py. Aísla el efecto de la ventana sin cambiar las columnas.
ventana141 = pd.read_csv("resultados/comparacion_ventana141.csv")
display(ventana141[["config", "n_features", "auc_mean", "spec_S0", "recall_5"]].round(3))


,config,n_features,auc_mean,spec_S0,recall_5
0,crudo 1 s (paper),141,0.862,0.957,0.352
1,rango 20 s,141,0.872,0.441,0.464
2,rango 30 s,141,0.864,0.430,0.423
3,rango 60 s,141,0.847,0.464,0.384
4,rango 120 s,141,0.874,0.536,0.458


La tabla deja tres cosas claras.

1. **La especificidad de 0,957 en el crudo no es transferencia: es el reloj.** Los 141 canales incluyen `Substation-AC-Meter.realtime`, un tiempo absoluto en epoch que separa S0 por fecha. En el paper esa columna también entra, y viaja dentro de cada ventana de 20 pasos.
2. **El rango móvil no empeora al modelo: le saca el atajo.** En el crudo la columna conserva el valor absoluto del reloj y con eso la fecha del escenario. Con rango móvil cada fila guarda solo la variación del último tramo, y la fecha deja de estar disponible; el número de columnas que llega al modelo es el mismo en los dos casos (132, porque se descartan las 9 constantes). La especificidad de 0,441 mide la fuga que desapareció, no un modelo peor.
3. **Entre 20 s y 120 s, el AUC y el recall quedan casi empatados** (0,872 contra 0,874 y 0,464 contra 0,458), pero la especificidad en S0 no: 0,441 contra 0,536. Y el orden deja de ser monótono (60 s es el más flojo). La elección de 120 s se sostiene por la especificidad, no por el AUC.

Lo que esta tabla **no** responde: si conviene o no la ventana de 20 pasos del paper. Eso se replica en §3.5.


### La prueba que faltaba: `zscore` tiene mejor AUC

La tabla de métodos de la sección 1 deja una objeción obvia: **`zscore` tiene el
AUC más alto (0,900 contra 0,880)**. Si se ordena por AUC, la decisión de usar
rango móvil parece arbitraria.

La respuesta es que el AUC no es la métrica operativa. La celda siguiente mide,
por método y con el mismo modelo y protocolo LOSO de `src/sweep.py`:

| métrica | qué prueba |
|---|---|
| `celdas vacías` | `zscore` divide por el desvío de la ventana y lo reemplaza por `NaN` cuando es cero |
| `espec. S0` / `marca S0` | transferencia y sobre-marcado sobre el benigno no visto |
| `recall@5%` | detección con el mismo presupuesto de falsas alarmas |
| `AUC desvío` | estabilidad del AUC entre folds |
| `desvío de ventana por escenario` | control exploratorio del mecanismo |

El último control se mide para no afirmar un mecanismo sin evidencia. Si no se
sostiene, se dice.


In [8]:
# Justificación cuantitativa de la decisión: robustez de cada transformación
# causal a 120 s, con el mismo modelo y protocolo que src/sweep.py.
from src.sweep import _sweep_model
from src.compare_paper import _fold_metrics
from src.pipeline import build_pipeline, balanced_weights

METODOS = ("none", "mean", "zscore", "std", "range")


def evaluar_metodo(nombre, ventana=120):
    Xt = detrend(X_crudo, escenario, method=nombre, window=ventana)
    aucs, recalls = [], []
    espec = marca = pmedia = float("nan")
    for retenido in ESCENARIOS:
        test = (escenario == retenido).to_numpy()
        train = ~test
        pipe = build_pipeline(_sweep_model()).fit(
            Xt.loc[train], y.loc[train], model__sample_weight=balanced_weights(y.loc[train])
        )
        score = pipe.predict_proba(Xt.loc[test])[:, 1]
        met = _fold_metrics(y.loc[test].to_numpy(), score)
        if np.isfinite(met["auc"]):
            aucs.append(met["auc"])
            recalls.append(met["recall_at_5"])
        if retenido == "S0":
            espec = met["specificity"]
            marca = float((score >= 0.5).mean())
            pmedia = float(score.mean())
    return {
        "método": nombre,
        "AUC medio": float(np.mean(aucs)),
        "AUC desvío": float(np.std(aucs)),
        "espec. S0": espec,
        "marca S0": marca,
        "p media S0": pmedia,
        "recall@5%": float(np.nanmean(recalls)),
        "celdas vacías": float(Xt.isna().to_numpy().mean()),
    }


justificacion = pd.DataFrame([evaluar_metodo(m) for m in METODOS]).set_index("método")
justificacion.to_csv("resultados/justificacion_detrend.csv")
display(justificacion.round(4))

# Mecanismo: desvío de ventana medio por escenario (el denominador de zscore).
desvio_ventana = (
    X_crudo.groupby(escenario)
    .rolling(120, min_periods=1)
    .std(ddof=0)
    .reset_index(level=0, drop=True)
    .groupby(escenario)
    .mean()
    .mean(axis=1)
    .rename("desvío medio de ventana")
)
desvio_ventana.to_frame().round(4).to_csv("resultados/detrend_std_escenario.csv")
display(desvio_ventana.round(4).to_frame())


,AUC medio,AUC desvío,espec. S0,marca S0,p media S0,recall@5%,celdas vacías
método,,,,,,,
none,0.8245,0.0791,0.4549,0.5451,0.4818,0.2264,0.0023
mean,0.8862,0.0821,0.4661,0.5339,0.5186,0.4207,0.0023
zscore,0.9003,0.0381,0.4188,0.5812,0.4991,0.4473,0.0118
std,0.8933,0.0723,0.5017,0.4983,0.4809,0.4360,0.0023
range,0.8805,0.0745,0.5228,0.4772,0.4876,0.5237,0.0023


,desvío medio de ventana
scenario,
S0,478.1252
S1,418.8054
S1alt,343.6979
S2,547.1094
S3,468.4382
S4,454.6769
S5,479.7018
S6,464.8970


### El argumento

El intercambio medido, rango contra `zscore`, en la misma ventana de 120 s:

| | rango 120 s | `zscore` 120 s | diferencia |
|---|---|---|---|
| AUC medio | 0,880 | **0,900** | −0,020 |
| espec. S0 | **0,523** | 0,419 | +0,104 |
| recall@5 | **0,524** | 0,447 | +0,077 |
| celdas vacías | **0,23 %** | 1,18 % | +0,95 pp |
| AUC desvío | 0,074 | **0,038** | — |

> El rango móvil pierde 0,020 de AUC y gana 0,104 de especificidad y 0,077 de
> recall@5 sobre el benigno no visto, y además degenera menos. Como la métrica
> operativa es no marcar benigno sin perder detección, el intercambio es
> favorable.

Dos honestidades:

- **La estabilidad no es el argumento.** `zscore` tiene el AUC más estable entre
  folds (0,038 contra 0,074). No lo usamos como razón.
- **El mecanismo simple no se sostuvo.** Esperábamos que S0 fuera el escenario
  más "quieto" y que el denominador de `zscore` explotara ahí. El desvío medio de
  ventana de S0 (478) no es el más bajo (S1alt 344, S1 419). La peor calibración
  de `zscore` sobre S0 es un **hecho medido**, pero su causa queda como pregunta
  abierta, no como afirmación.

Si los números de la celda anterior no acompañan esa tabla, **se reporta y se
revisa la decisión**, no se cambia el argumento para que gane.


### Los límites de la decisión

- El rango móvil quita el **nivel**, no la **estructura temporal**: la
  correlación con el reloj (~0,23) es la misma que en el crudo.
- La especificidad en S0 queda entre 0,42 y 0,52 en todos los métodos. El
  detrend es **condición necesaria, no suficiente** para transferir: el límite
  es el dataset, no la transformación.
- En el barrido de 141 canales el AUC queda plano entre ventanas (0,847–0,874) y
  la especificidad es lo que separa; en el de `block_pca` crecen las dos, y la
  especificidad lo hace en mayor medida (0,445 → 0,503).


## 2. block_pca no reemplaza al detrend

De la comparación anterior quedan dos preguntas naturales:

1. **¿`block_pca` corrige el crudo?** Es una reducción de columnas, no un `detrend`. Quizás, al comprimir cada bloque con PCA, el nivel crudo (el reloj) queda repartido entre componentes y deja de dañar la transferencia.
2. **¿Y si sacamos `Profile.*`?** Son las columnas que sospechamos del nivel fuera de rango en S0 (irradiancia, carga de base). Eliminarlas debería arreglar la especificidad, si la fuga viene de ahí.

La celda lee `resultados/blockpca_ablacion.csv`, que corre LOSO con `block_pca` y Random Forest en las cuatro combinaciones: con/sin rango móvil, con/sin `Profile.*`.

In [9]:
ablacion = pd.read_csv("resultados/blockpca_ablacion.csv")

etiquetas = {
    "block_pca + rango (estándar)": ("con Profile", "rango móvil"),
    "block_pca + crudo (paper, 1 s)": ("con Profile", "crudo (paper)"),
    "block_pca + rango sin Profile": ("sin Profile", "rango móvil"),
    "block_pca + crudo sin Profile": ("sin Profile", "crudo (paper)"),
}
ablacion["profile"] = ablacion["config"].map(lambda c: etiquetas[c][0])
ablacion["transformacion"] = ablacion["config"].map(lambda c: etiquetas[c][1])

espec = ablacion.pivot(index="profile", columns="transformacion", values="spec_S0")
auc = ablacion.pivot(index="profile", columns="transformacion", values="auc_mean")
recall = ablacion.pivot(index="profile", columns="transformacion", values="recall_5")

print("especificidad en S0 (umbral 0,5): más alto = menos falsas alarmas")
display(espec.round(3))
print("AUC medio (folds con ataque)")
display(auc.round(3))
print("recall@5% FPR")
display(recall.round(3))

especificidad en S0 (umbral 0,5): más alto = menos falsas alarmas


transformacion,crudo (paper),rango móvil
profile,,
con Profile,0.478,0.815
sin Profile,0.435,0.507


AUC medio (folds con ataque)


transformacion,crudo (paper),rango móvil
profile,,
con Profile,0.900,0.952
sin Profile,0.874,0.902


recall@5% FPR


transformacion,crudo (paper),rango móvil
profile,,
con Profile,0.411,0.752
sin Profile,0.367,0.527


La lectura:

1. **`block_pca` no corrige el crudo.** Con crudo la especificidad en S0 queda en 0,435–0,478; con rango móvil sube a 0,507. `block_pca` comprime los bloques con PCA pero no elimina el nivel: lo reparte entre componentes. Reducción ≠ `detrend`.

2. **La tabla muestra, de nuevo, la fuga ya declarada arriba.** Incluir `Profile.*` infla la especificidad en S0 (0,815 contra 0,507) porque el perfil está congelado en S0. Ese 0,815 era identidad de escenario, no transferencia. Por eso la receta final usa el rango móvil sin `Profile.*` y reporta el 0,507 como el número honesto.

3. **El `detrend` es necesario pero no suficiente.** Pasar de crudo a rango móvil sube la especificidad (0,435 → 0,507 sin `Profile.*`), pero la transferencia a S0 sigue pobre. Ninguna selección de columnas reemplaza el problema de fondo: el nivel y la estructura temporal entre escenarios.

## 3. Comparación con los dos papers

Acá hay dos papers sobre el mismo dataset y conviene no mezclarlos:

- El **paper del dataset** (Data4Cyber, §V-F, tabla IV) evalúa un subconjunto fijo de **ocho features** —potencia activa de BSS, PV, Load A, Load B y subestación, SoC de referencia, SoC reportado y precio MQTT— con un **corte cronológico bloqueado**. Reporta precisión, recall, F1 y tasa de falsos positivos agregadas; no reporta AUC.
- **GRIDGUARD-AI** (tabla II) evalúa **141 canales** con **LOSO** y reporta un ROC-AUC por escenario retenido. Su unidad de evaluación es una **ventana de 20 pasos × 141 variables** (14.202 ventanas): el TCN consume la secuencia y el RF entrega un score tabular sobre esa ventana.

Los replicamos por separado, siempre con el mismo Random Forest (120 árboles, profundidad 18, clases balanceadas, hoja mínima 2).

La réplica de GRIDGUARD-AI es la de la **sección 4**: su unidad es la ventana de 20 pasos, y así se replica. Lo que sigue acá es la tabla IV y un **diagnóstico** sobre los 141 canales clasificando fila suelta, que sirve para aislar el efecto del reloj pero **no es** la réplica de su tabla II.

**Tabla IV.** Mismo recorte que el paper: ocho features, el primer 70 % de cada escenario para entrenar y el último 30 % para probar, con un hueco de 19 filas. La réplica reproduce la patología que el paper describe: recall altísimo con casi una falsa alarma por cada benigno.

In [10]:
tabla_iv = pd.read_csv("resultados/comparacion_table_iv.csv").set_index("config")
tabla_iv.columns = ["precisión", "recall", "F1", "FPR"]
display(tabla_iv.round(3))

,precisión,recall,F1,FPR
config,,,,
"paper (RF, Table IV)",0.746,0.796,0.771,0.858
"replica (8 features, split bloqueado)",0.689,0.991,0.813,0.907


> **Atribución.** Las columnas «paper RF (tabla II)» y «paper TCN (tabla II)» son los valores **publicados**, transcriptos tal cual: S0 y S3 N/A por ser folds de una sola clase. No son nuestros.

**Tabla IV y diagnóstico de 141 canales.** La tabla IV se replica con su propio recorte: ocho features, 70/30 cronológico con hueco de 19 filas. El bloque de 141 canales que sigue **no es la réplica de la tabla II**: clasifica fila suelta sobre la misma matriz del paper, y se incluye para aislar el efecto de las columnas. La réplica con su unidad —ventana de 20 pasos— está en la sección 4.

El set de 141 canales incluye `Substation-AC-Meter.realtime`, que no es una medición eléctrica sino un **reloj absoluto**: monótono dentro de cada escenario y con correlación 1,0 con la posición de la fila. El baseline del paper incluye ese reloj, además de la telemetría.

El diagnóstico por fila suelta da **0,857** de media (0,79–0,89 por fold). Coincide con el paper en que **S6 es el fold más fuerte** (0,894 contra 0,953) y **S2 el más débil** (0,795 contra 0,727). No coincide en S1: el paper reporta 0,346, por debajo del azar; nosotros obtenemos 0,871. No pudimos reconstruir ese fold y queda como desconocido.

Ese 0,857 **no se compara con el 0,760 publicado**: nuestro diagnóstico entrena con todos los escenarios restantes, mientras que el paper entrena con una porción cronológica (ver la nota de protocolo en la sección 5). La comparación que corresponde es la de la sección 4, con su unidad.

Bajo el mismo LOSO y sobre filas sueltas, nuestras tres configuraciones ordenan así:

- **8 features, 0,934**: incluye el precio MQTT, que en S6 es la columna que el ataque escribe (fuga), y las dos columnas de SoC.
- **141 canales, 0,857**: el set del paper con su reloj adentro.
- **block_pca 120 s, 0,907**: nuestra receta sobre las 132 columnas sin fuga. No es una réplica de la tabla II; es otra representación.

S3 no entra en la media: el paper lo reporta como fold de una sola clase. En nuestros datos S3 tiene 912 ataques y 888 benignos, así que podemos calcular 0,895, pero lo dejamos afuera para comparar contra la misma tabla.


In [11]:
gridguard = pd.read_csv("resultados/comparacion_gridguard.csv").set_index("fold")
gridguard = gridguard.rename(columns={
    "paper_rf": "paper RF (tabla II)",
    "paper_tcn": "paper TCN (tabla II)",
    "paper8_auc": "nuestro RF, 8 features (tabla IV)",
    "all141_auc": "nuestro RF, 141 canales (fila suelta)",
    "blockpca_auc": "nuestro RF, block_pca 120 s (fila suelta)",
})
evaluables = ["S1", "S1alt", "S2", "S4", "S5", "S6"]
print("ROC-AUC por fold retenido (diagnóstico por fila suelta)")
display(gridguard.loc[evaluables].round(3))
print("media de los seis folds evaluables")
display(gridguard.loc[evaluables].mean().round(3).to_frame("AUC medio").T)


ROC-AUC por fold retenido (diagnóstico por fila suelta)


,paper RF (tabla II),paper TCN (tabla II),"nuestro RF, 8 features (tabla IV)","nuestro RF, 141 canales (fila suelta)","nuestro RF, block_pca 120 s (fila suelta)"
fold,,,,,
S1,0.346,0.162,0.940,0.871,0.852
S1alt,0.807,0.756,0.891,0.874,0.820
S2,0.727,0.673,0.892,0.795,0.917
S4,0.839,0.517,0.979,0.861,0.983
S5,0.886,0.815,0.987,0.848,0.946
S6,0.953,0.812,0.918,0.894,0.921


media de los seis folds evaluables


,paper RF (tabla II),paper TCN (tabla II),"nuestro RF, 8 features (tabla IV)","nuestro RF, 141 canales (fila suelta)","nuestro RF, block_pca 120 s (fila suelta)"
AUC medio,0.76,0.622,0.934,0.857,0.907


### Más métricas de la réplica (141 canales)

El paper publica solo AUC por escenario para el RF. Con el mismo modelo y las mismas 141 columnas podemos reportar la batería completa por fold —precisión, recall, F1, especificidad, balanced accuracy y recall a presupuestos de falsa alarma—, que es lo que hace falta para hablar de operación y no solo de ranking. Sale de `resultados/metricas_rf.csv`, que genera el mismo `python -m src.compare_paper`.

Dos lecturas. Primero, el AUC ordena pero no fija el umbral: a 0,5 el recall de S1 es 0,02 aunque su AUC sea 0,87. Segundo, la especificidad en S0 es 0,957 para los 141 canales contra 0,503 de `block_pca`; no es mejor transferencia, es el reloj (`Substation-AC-Meter.realtime`) que trae el set del paper y que separa S0 por fecha.

Nuestra réplica clasifica filas sueltas, no ventanas de 20 pasos. Restringirla a las 14.202 filas que cierran cada ventana del paper (descartando las primeras 19 de cada escenario) mueve cada fold menos de 0,012 y la media de 0,8574 a 0,8569, así que esa diferencia de muestreo mueve poco el AUC. La ventana aplanada, que es lo que hace el paper, se evalúa en §3.5.


In [12]:
metricas = pd.read_csv("resultados/metricas_rf.csv")
replica_141 = (
    metricas[metricas.config == "141 canales (paper)"]
    .set_index("fold")
    .drop(columns="config")
    .rename(columns={
        "n_pos": "ataques", "n_neg": "benignos", "auc": "AUC", "pr_auc": "PR-AUC",
        "precision": "precisión@0.5", "recall": "recall@0.5", "f1": "F1@0.5",
        "specificity": "espec.@0.5", "balanced_acc": "bal. acc",
        "recall_at_1": "recall@1%", "recall_at_5": "recall@5%", "recall_at_10": "recall@10%",
    })
)
display(replica_141.round(3))


,ataques,benignos,AUC,PR-AUC,precisión@0.5,recall@0.5,F1@0.5,espec.@0.5,bal. acc,recall@1%,recall@5%,recall@10%
fold,,,,,,,,,,,,
S0,0,1798,NaN,NaN,NaN,NaN,NaN,0.957,NaN,NaN,NaN,NaN
S1,913,885,0.871,0.874,1.000,0.018,0.034,1.000,0.509,0.313,0.486,0.560
S1alt,910,882,0.874,0.842,0.841,0.186,0.304,0.964,0.575,0.133,0.262,0.512
S2,903,865,0.795,0.790,0.758,0.470,0.580,0.844,0.657,0.189,0.239,0.339
S3,912,888,0.889,0.853,0.847,0.122,0.213,0.978,0.550,0.011,0.468,0.714
S4,1257,541,0.861,0.935,0.936,0.591,0.724,0.906,0.748,0.417,0.417,0.591
S5,1341,459,0.848,0.876,0.904,0.430,0.582,0.867,0.648,0.011,0.011,0.222
S6,1350,450,0.894,0.943,0.965,0.430,0.595,0.953,0.692,0.007,0.578,0.770
pooled,7586,6768,0.832,0.833,0.868,0.345,0.494,0.941,0.643,0.133,0.331,0.521


## 4. Réplica de GRIDGUARD-AI: su unidad, la ventana de 20 pasos

**Esta es la réplica que corresponde a su tabla II.** El diagnóstico por fila suelta de la sección 3 sirve para aislar el efecto del reloj, no para replicar el paper. Acá armamos la ventana como la describe: **20 pasos × 141 variables por muestra**, con paso 1, descartando las primeras 19 filas de cada escenario. Da 14.202 ventanas, el mismo número que reporta la tabla I. Cada ventana se aplana a 20 × 141 = 2.820 columnas y el RF del paper la clasifica como una fila más.

Dos decisiones que el paper no explicita, y cómo las resolvimos:

- **Etiqueta de la ventana**: hereda la de su fila de cierre. Con voto de mayoría cambiarían 204 ventanas de 14.202 (1,4 %), así que la regla no mueve el resultado.
- **Cómo consume el RF la ventana**: el paper solo dice "supervised tabular decision score", así que la aplanamos. El TCN consume la secuencia (20, 141) y queda fuera de alcance.

Imputación por mediana y filtro de varianza se ajustan en cada fold de entrenamiento; el escalado se omite porque no cambia nada en un árbol. Sale de `python -m src.paper_windows`. Como el set del paper incluye `Substation-AC-Meter.realtime`, corrimos además la variante sin las cinco columnas `.realtime` (`--drop-realtime`), que guarda `resultados/ventana20_sin_reloj_loso.csv`.


In [ ]:
# La ventana del paper, construida sobre los datos: 20 pasos consecutivos,
# stride 1, dentro de cada escenario. Se descartan las primeras 19 filas de cada
# uno, así que cada escenario aporta (filas - 19) ventanas.
filas_por_escenario = escenario.value_counts().reindex(ESCENARIOS)
ventanas_por_escenario = filas_por_escenario - 19
tabla_ventanas = pd.DataFrame({
    "filas": filas_por_escenario,
    "ventanas (filas - 19)": ventanas_por_escenario,
})
display(tabla_ventanas)
print("total de ventanas:", int(ventanas_por_escenario.sum()), "| el paper reporta 14.202")
print("cada ventana:", (20, 141), "-> aplanada:", 20 * 141, "columnas")


,filas,ventanas (filas - 19)
scenario,,
S0,1798,1779
S1,1798,1779
S1alt,1792,1773
S2,1768,1749
S3,1800,1781
S4,1798,1779
S5,1800,1781
S6,1800,1781


total de ventanas: 14202 | el paper reporta 14.202
cada ventana: (20, 141) -> aplanada: 2820 columnas


In [13]:
ventana = pd.read_csv("resultados/ventana20_loso.csv").set_index("fold")
sin_reloj = pd.read_csv("resultados/ventana20_sin_reloj_loso.csv").set_index("fold")
gridguard = pd.read_csv("resultados/comparacion_gridguard.csv").set_index("fold")

evaluables = ["S1", "S1alt", "S2", "S4", "S5", "S6"]
comparacion_ventana = pd.DataFrame({
    "paper RF (tabla II)": gridguard["paper_rf"],
    "paper TCN (tabla II)": gridguard["paper_tcn"],
    "réplica 20x141 con reloj": ventana["auc"],
    "réplica 20x141 sin .realtime": sin_reloj["auc"],
}).loc[evaluables].round(3)
display(comparacion_ventana)
display(comparacion_ventana.mean().round(3).to_frame("AUC medio").T)


,paper RF (tabla II),paper TCN (tabla II),réplica 20x141 con reloj,réplica 20x141 sin .realtime
fold,,,,
S1,0.346,0.162,0.979,0.514
S1alt,0.807,0.756,0.950,0.909
S2,0.727,0.673,0.878,0.786
S4,0.839,0.517,0.858,0.871
S5,0.886,0.815,0.894,0.738
S6,0.953,0.812,0.830,0.878


,paper RF (tabla II),paper TCN (tabla II),réplica 20x141 con reloj,réplica 20x141 sin .realtime
AUC medio,0.76,0.622,0.898,0.783


### Nota de protocolo (verificada contra el paper)

`gridguard.md`, el texto del paper que está en el repo, permite verificar el
protocolo en vez de suponerlo: ventanas de 20 pasos con stride 1 → 14.202
ventanas (14.354 − 8 × 19); LOSO de 8 folds con el escenario retenido excluido;
y, **dentro de los escenarios restantes, partición cronológica fit/validación con
purga de 19 pasos**. El RF es de 120 árboles, profundidad 18, hoja mínima 2 y
clases balanceadas.

Nuestra réplica coincide en los hiperparámetros (`paper_rf`) y en la partición
bloqueada (`_blocked_split`, cronológica 70/30 con purga 19). **No coincide en un
punto**: `_loso_scores` y `src/paper_windows.py` entrenan con todos los escenarios
restantes, sin el split cronológico interno. Eso le da al modelo más datos que al
paper. Es un candidato concreto para explicar la inversión de S1 y la ventaja de
S6, pero **todavía no está medido**: hasta correrlo, siguen siendo desconocidos.


## 5. Qué se puede afirmar, y qué no

La comparación tiene tres niveles y conviene no mezclarlos.

**Contra su configuración de 141 canales (tabla II).** Bajo el mismo LOSO,
nuestra réplica a nivel fila da **0,857** contra **0,760** publicado. Gana en S1,
S1alt, S2 y S4; pierde en S5 (0,848 contra 0,886) y S6 (0,894 contra 0,953). El
fold S1 no se reproduce: el paper reporta 0,346 y nuestra réplica 0,871. Es el
desacuerdo más grande y queda como desconocido.

**Contra su unidad de evaluación (ventana de 20 pasos).** Aplanada, la ventana da
**0,898 con** el reloj adentro y **0,783 sin** `.realtime`. La primera fila es la
**ablación**, no una réplica: el 0,898 mide la fuga del reloj, que viaja veinte
veces por muestra. La segunda **coincide en magnitud** con el 0,760 publicado
(Δ = +0,023) y es la lectura que sostenemos.

**Nuestra receta con `block_pca`.** Da **0,907** sobre las 132 columnas sin fuga.
Eso **no mide al paper**: es otra representación. Sirve como comparación interna
contra nuestra propia réplica de 141 canales (0,907 contra 0,857), no como
superación. Tampoco alcanza al baseline de 8 features (0,934), que usa el precio
MQTT y el SoC.

Lo que ninguna configuración demuestra:

1. **Detección libre del tiempo.** Un control que solo usa la posición relativa
   dentro del escenario ronda **0,96** de AUC medio (`resultados/binario.csv`),
   más alto que el modelo completo (0,883). El reloj sigue siendo la explicación
   más simple.
2. **Transferencia al benigno no visto.** Las configuraciones sin reloj quedan
   entre 0,47 y 0,54 de especificidad en S0. Las que incluyen el reloj suben a
   0,96–0,99 por la fuga, no por detección.
3. **Que el paper haya quitado el reloj.** No lo dice. Sus números son
   compatibles con una vista sin él, nada más.

El aporte no es ganar por AUC: es mostrar, con el mismo dato y el mismo modelo,
de dónde sale el AUC que ambos reportan.


## Cierre de la rama

Lo que esta rama deja establecido:

1. La transformación se elige por especificidad en S0, y el rango móvil de 120 s
   es la mejor de las opciones causales bajo ese criterio.
2. El rango móvil no limpia el reloj: quita el nivel. La ventaja de AUC del
   `zscore` no compensa su peor transferencia.
3. Contra los papers, la comparación honesta es "coincide en magnitud sin el
   reloj", no "supera al paper".

Volver a la columna vertebral: [`02_supervisado.ipynb`](02_supervisado.ipynb).
Otras ramas: [`02b_reduccion.ipynb`](02b_reduccion.ipynb),
[`02c_hiperparametros.ipynb`](02c_hiperparametros.ipynb).
